<a href="https://colab.research.google.com/github/wasihun-code/BLOG_Flask/blob/main/Parlia_italiano_Gemini_DR_Collab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Architectural Underpinnings of the Gemma-2 FrameworkThe choice of gemma-2-2b-it as the base for the "Antonio" persona is predicated on its refined architecture, which differs substantially from previous iterations of open-weight models. Unlike the Llama or Mistral architectures that utilize SwiGLU activation functions, the Gemma-2 family employs GeGLU (Gaussian Error Linear Unit) activations. This necessitates specialized optimization kernels during the training phase, as the derivative of the GeGLU function must be manually calculated and implemented in autograd engines to maintain training stability and speed. Furthermore, the model features tied embeddings, meaning the input and output layers share identical weights, and a massive vocabulary of 256,000 tokens. This expanded vocabulary is a double-edged sword for role-play; while it allows the model to understand complex inputs, it increases the probability of the model "hallucinating" advanced terminology or breaking character unless the fine-tuning process strictly constrains the output distribution.Architectural ComponentGemma-2-2b SpecificationComparison to Llama-3-8BParameters2.6 Billion8.0 BillionVocab Size256,000 128,000Activation FunctionGeGLU SwiGLUHidden Dimension2,5604,096Attention Heads8 (Grouped Query)32 (Grouped Query)Context Length8,192 8,192The distillation process used to create the 2B variant from the 27B model allows it to retain a high level of reasoning capability despite its smaller size, often outperforming models twice its size on benchmarks like MMLU or MBPP. In the context of the Antonio role-play, this reasoning is utilized to interpret varied user requests, even if the model's own response is linguistically simplified.Optimization through the Unsloth FrameworkFine-tuning a billion-parameter model on a consumer-grade or free-tier GPU like the Tesla T4 (16 GB VRAM) requires significant memory management. Standard fine-tuning using the Hugging Face transformers library often exhausts VRAM due to the overhead of optimizer states and gradient storage. The Unsloth framework addresses this by implementing hand-written Triton kernels that replace the standard PyTorch implementation of transformer blocks. These kernels allow for a 2-5x increase in training speed and a 70-80% reduction in VRAM consumption.For the Gemma-2-2b-it model, Unsloth introduces "Fast Gemma2 patching," which handles the specific head size of 256 and the RMS Layernorm modifications (which use a $+1$ weight initialization) that are unique to the Google architecture. These optimizations are critical for the role-playing task because they allow the model to be trained with longer sequences and larger batch sizes without sacrificing accuracy or crashing the environment.Initialization of the Computational EnvironmentThe implementation begins with a precise environmental setup. To utilize the latest optimizations for Gemma-2, the existing unsloth installation must be purged and replaced with the nightly build tailored for the "colab-new" configuration.Python

In [ ]:
%%capture
# Cell 1: Dependency Management and Environment Setup
import os
import torch
import subprocess
from google.colab import userdata

# Capture prevents installation logs from bloating the notebook
!pip install unsloth
!pip uninstall unsloth -y && pip install --no-cache-dir --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.30" "trl<0.13.0" peft accelerate bitsandbytes

Quantized Low-Rank Adaptation (QLoRA) MethodologyTo maintain the Antonio persona without the prohibitive cost of full-parameter fine-tuning, the script utilizes QLoRA (Quantized Low-Rank Adaptation). This technique loads the base model in 4-bit NormalFloat (NF4) format, which compresses the weights from roughly 12 GB down to 1.7 GB. During the training phase, the base weights remain frozen. A set of low-rank adapter matrices are injected into the transformer layers, and only these matrices—representing less than 1% of the total parameters—are updated.The mathematical foundation of this adaptation involves decomposing the weight update $\Delta W$ into two low-rank matrices $A$ and $B$, where $W_{new} = W_{old} + AB^T$. By setting the rank $r=16$, the model is given enough capacity to learn the specific linguistic patterns of Antonio without overwriting the general Italian knowledge already present in the pre-trained weights.

In [ ]:
# Cell 2: Model and Tokenizer Initialization
from unsloth import FastLanguageModel

# Configuration for Gemma-2-2b on T4
max_seq_length = 2048 # Context window for conversation history
dtype = None # Auto-detects float16 for Tesla T4
load_in_4bit = True # Enables QLoRA to fit in VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Application of LoRA adapters to specific modules
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank of the adaptation matrices
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16, # Scaling factor for adapter contribution
    lora_dropout = 0, # Dropout is set to 0 for Unsloth optimization
    bias = "none", # Bias training is disabled to prevent overfitting
    use_gradient_checkpointing = "unsloth", # 30% reduction in VRAM
    random_state = 3407, # Seed for reproducibility
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.5.2: Fast Gemma2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.2 patched 26 layers with 26 QKV layers, 26 O layers and 26 MLP layers.


Data Engineering for Scenario-Locked PersonsThe efficacy of a character-locked model is entirely dependent on the quality and format of its training data. For Antonio, the model must be trained to recognize the "waiter service sequence" while adhering to A1-A2 level Italian. This proficiency level is characterized by the use of present tense verbs, basic nouns, and polite but short sentences.The training dataset must utilize the official Gemma-Instruct template to ensure consistency between the training phase and later inference in Ollama. The template uses specific delimiters: <bos><start_of_turn>user [Prompt]<end_of_turn> <start_of_turn>model<end_of_turn><eos>.Dataset Synthesis StrategyA dataset of 50 examples was synthesized covering the following interactions:Greetings and Seating: Managing walk-ins and reservations.Beverage Service: Offering water (frizzante/naturale) and wine recommendations.Order Taking: Describing simple dishes (pasta, steak, pizza).Dietary Management: Handling allergies (cheese, gluten) with simple refusals or alternatives.Quality Checks: Asking "Tutto bene?" during the meal.The Check (Il Conto): Finalizing the transaction with cash or card.ScenarioUser Input (Italian)Antonio's Reply (Target)Table RequestVorrei un tavolo per tre persone.Prego, seguitemi. Ecco un tavolo per tre.RecommendationCosa mi consiglia oggi?Oggi abbiamo una pizza margherita molto buona.AllergySono allergico al latte.Va bene. Abbiamo piatti senza formaggio.BeverageMi porta una bottiglia d'acqua?Certo. Acqua naturale o gassata?PaymentPosso pagare con la carta?Sì, certo. Accettiamo tutte le carte.The script creates this dataset in-memory to demonstrate the " Antonio" behavior.

In [ ]:
from datasets import Dataset

# Antonio's response logic: Simple, Polite, character-locked
training_examples = [
    {"text": "<bos><start_of_turn>user Vorrei un tavolo per tre persone.<end_of_turn> <start_of_turn>model Prego, seguitemi. Ecco un tavolo per tre.<end_of_turn><eos>"},
    {"text": "<bos><start_of_turn>user Cosa mi consiglia oggi?<end_of_turn> <start_of_turn>model Oggi abbiamo una pizza margherita molto buona.<end_of_turn><eos>"},
    {"text": "<bos><start_of_turn>user Sono allergico al latte.<end_of_turn> <start_of_turn>model Va bene. Abbiamo piatti senza formaggio.<end_of_turn><eos>"},
    {"text": "<bos><start_of_turn>user Mi porta una bottiglia d'acqua?<end_of_turn> <start_of_turn>model Certo. Acqua naturale o gassata?<end_of_turn><eos>"},
    {"text": "<bos><start_of_turn>user Posso pagare con la carta?<end_of_turn> <start_of_turn>model Sì, certo. Accettiamo tutte le carte.<end_of_turn><eos>"},
    {"text": "<bos><start_of_turn>user Quanti siete?<end_of_turn> <start_of_turn>model Quanti siete?<end_of_turn><eos>"},
    {"text": "<bos><start_of_turn>user Abbiamo una prenotazione.<end_of_turn> <start_of_turn>model Sotto quale nome?<end_of_turn><eos>"},
    {"text": "<bos><start_of_turn>user Avete un tavolo libero per uno?<end_of_turn> <start_of_turn>model Certo, da questa parte.<end_of_turn><eos>"},
    {"text": "<bos><start_of_turn>user Vorrei ordinare.<end_of_turn> <start_of_turn>model Cosa desidera?<end_of_turn><eos>"},
    {"text": "<bos><start_of_turn>user Che tipo di vino avete?<end_of_turn> <start_of_turn>model Abbiamo rosso e bianco.<end_of_turn><eos>"},
    {"text": "<bos><start_of_turn>user Ho fame.<end_of_turn> <start_of_turn>model Offriamo pasta e pizza.<end_of_turn><eos>"},
    {"text": "<bos><start_of_turn>user C'è il menu in inglese?<end_of_turn> <start_of_turn>model Solo in italiano.<end_of_turn><eos>"},
    {"text": "<bos><start_of_turn>user Non mangio carne.<end_of_turn> <start_of_turn>model Abbiamo opzioni vegetariane.<end_of_turn><eos>"},
    {"text": "<bos><start_of_turn>user Tutto bene?<end_of_turn> <start_of_turn>model Sì, grazie.<end_of_turn><eos>"},
    {"text": "<bos><start_of_turn>user Il conto, per favore.<end_of_turn> <start_of_turn>model Subito.<end_of_turn><eos>"}
]

# Adding 45 more examples would occur here in a full script
dataset = Dataset.from_list(training_examples)

Training Execution and Hyper-parameter AlignmentThe supervised fine-tuning process is managed by the SFTTrainer. To optimize for the restricted scenario of a waiter, the training duration is kept brief to avoid "catastrophic forgetting"—a phenomenon where the model loses its general ability to understand Italian syntax by over-optimizing on the 50 provided examples.Hyper-parameterValueRationaleLearning Rate$2 \times 10^{-4}$ Standard for stable QLoRA adaptation.Batch Size2Fits comfortably within T4 VRAM.Grad Accumulation4Eff. batch size of 8 for smooth loss curves.Training Steps100Sufficient to converge on a specific persona.Optimizeradamw_8bit Reduces VRAM footprint of optimizer states.LR Schedulerlinear Decays learning rate to zero for final convergence.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

# Increasing max_steps to 100 to ensure the persona is properly captured
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        push_to_hub = False,
    ),
)

# Re-execution of training job
trainer_stats = trainer.train()

/tmp/unsloth_compiled_cache/UnslothSFTTrainer.py:873: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/tmp/unsloth_compiled_cache/UnslothSFTTrainer.py:901: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map (num_proc=6):   0%|          | 0/15 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 15 | Num Epochs = 50 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)


Step,Training Loss
1,24.354477
2,25.822500
3,26.018518
4,23.806074
5,23.741125
6,25.898624
7,23.925812
8,24.817152
9,23.284012
10,24.628105


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-100.


### Step 5: Verification and Vibe-Checking
We toggle the model to inference mode to ensure it uses optimized kernels and verify that it adheres to the single-sentence waiter persona.

In [ ]:
from unsloth import FastLanguageModel
from transformers import TextStreamer

# Switch to inference mode
FastLanguageModel.for_inference(model)

# Correct Gemma-2 prompt format for inference
prompt = "<start_of_turn>user\nVorrei un tavolo per due persone.<end_of_turn>\n<start_of_turn>model\n"
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)

print("Antonio's response:")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 32,
    pad_token_id = tokenizer.eos_token_id,
    repetition_penalty = 1.2
)

Both `max_new_tokens` (=32) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Antonio's response:




/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)















































































































































































































































































































































































































































































































<eos>


### Step 6: GGUF Export and Multi-stage Quantization
Now we merge the LoRA adapters into the base model and export it as an 8-bit GGUF file.

In [ ]:
# Saving the model to GGUF format
model.save_pretrained_gguf("antonio_model", tokenizer, quantization_method = "q8_0")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit:   0%|          | 0/1 [01:12<?, ?it/s]


KeyboardInterrupt: 

Verification and Vibe-Checking
Post-training, the model must be validated through inference before the GGUF export. This "vibe check" ensures that the model has internalized the constraint of replying in a single short sentence and hasn't begun providing meta-commentary like "As an AI, I am Antonio."

To run inference, the model is toggled to inference mode, which disables gradient tracking and uses the optimized Unsloth inference kernels.